## SDK 0.3.0 설치 / Install SDK 0.3.0
첫 셀은 SDK 0.3.0이 없을 때만 고정 wheel을 설치합니다(네트워크 필요). 설치된 경우 건너뜁니다. 결과 전송은 별도 선택이며 기본 False입니다.
The first cell installs the pinned wheel only if SDK 0.3.0 is absent (network required). If 0.3.0 is already installed, installation is skipped. Publishing results is a separate choice, default False.


In [ ]:
# Install only when this SDK version is absent. No result publication.
import importlib
import subprocess
import sys

try:
    import luna_repro
    current_sdk = getattr(luna_repro, "__version__", None)
except ModuleNotFoundError as error:
    if error.name != "luna_repro":
        raise
    current_sdk = None

if current_sdk != "0.3.0":
    subprocess.check_call([sys.executable, "-m", "pip", "install",
        "https://github.com/lunalab-ai/luna-repro-sdk/releases/download/v0.3.0/luna_repro-0.3.0-py3-none-any.whl#sha256=4fc3b6a0ff1d78dbb2ace0ea08dcedc3794e14f3930d0c60f29c42fbb04e2489"])
    if current_sdk is not None:
        raise RuntimeError("SDK updated. Restart the runtime/kernel, then Run all again. SDK 업데이트 후 런타임을 재시작하고 다시 실행하세요.")
    importlib.invalidate_caches()
    import luna_repro
assert luna_repro.__version__ == "0.3.0"
print("LUNA Repro SDK", luna_repro.__version__)


# One last cell / 마지막 한 셀
Synthetic CPU results only. Run all stays local. Connect and send are separate, disabled choices.
합성 CPU 결과입니다. 기본 전체 실행은 로컬 저장만 하며 연결과 전송은 각각 기본 꺼짐입니다.


In [ ]:
import luna_repro as repro
metrics = {"MAE": 0.5, "samples": 3}
comparison = [[0.0, 0.5], [1.0, 1.5], [2.0, 2.5]]
failed_check = {"passed": False, "maximum_error": 0.5}


For this offline demo, the UUID below is synthetic. For real work, use your permitted project UUID, then enable CONNECT explicitly and approve the browser code. No connection key is printed or saved.
아래 UUID는 로컬 실습용입니다. 실제 프로젝트 UUID로 바꾼 후 CONNECT를 명시적으로 켜고 브라우저에서 확인 코드를 승인하세요. 키는 출력하거나 파일에 저장하지 않습니다.


In [ ]:
DEMO_PROJECT = "12345678-1234-5678-1234-567812345678"
PROJECT = DEMO_PROJECT
CONNECT = False
if CONNECT:
    if PROJECT == DEMO_PROJECT:
        raise ValueError("Choose your actual permitted project UUID first")
    repro.connect(PROJECT, label="Colab", open_browser=False)
else:
    repro.configure(PROJECT)


In [ ]:
# Your own final cell can simply be: selected = repro.select()
selected = repro.select(namespace={"metrics": metrics, "comparison": comparison, "failed_check": failed_check},
    title="Synthetic selection / 합성 결과", purpose="error comparison", show=False)
selected.option("comparison", columns=["target", "predicted"])
# Set SHOW_SELECTOR=True in a notebook for checkboxes and separate Preview/Send buttons.
SHOW_SELECTOR = False
if SHOW_SELECTOR:
    selected.show()
selected.summary()


In [ ]:
from pathlib import Path
import uuid
output = Path(".luna-repro") / "quickstart" / uuid.uuid4().hex
selected.preview(output / "preview.html")
selected.save(output / "selected.json")
print("Local preview / 로컬 미리보기:", output / "preview.html")


Review the HTML or enable the notebook selector first. Only a published receipt confirms platform intake. Pending means the selected bundle remains in the local outbox.
HTML 또는 노트북 선택 화면에서 먼저 확인하세요. published 영수증만 서버 등록 완료이며 pending은 로컬 보존 상태입니다.


In [ ]:
PUBLISH = False
if PUBLISH:
    if not CONNECT or PROJECT == DEMO_PROJECT:
        raise ValueError("Connect the actual permitted project explicitly first")
    receipt = selected.publish()
    print(receipt)
